# Reuse the frozen LoD2 catalogue

Use Python 3.11+ and this installed package. This notebook loads the accepted physical catalogue offline, reuses its canonical definitions and inspects missing values. No files are written.

In [ ]:
from epsm_data import Catalogue, Layer, LayerMaterial

catalogue = Catalogue.load()
print(catalogue.summary())
assert len(catalogue.ids()) == 28
assert len(catalogue.search()) == 228
archetype = catalogue.for_year("multi_family", 1970, state="existing")
report = catalogue.completeness("mf_se02", state="existing")
print(report)

Loaded indexes are immutable references to shared definitions. Reuse an accepted material in a new project layer. The 0.15 m thickness is an illustrative project choice.

In [ ]:
material_id, material = next(iter(catalogue.materials.items()))
layer = Layer("Project layer", "project.layer", 0.15, (LayerMaterial(material, 1, 0.15),))
assert layer.materials[0].material is material
print(catalogue.material_records[material_id]["thermal_id"])
print(catalogue.material_records[material_id]["boverket_id"])
slot = catalogue.get_slot("multi_family:SE.02:SE.Zone3:existing")
assert catalogue.construction_sets[slot.construction_set_id] is archetype.construction_set

A project copy uses the standard authored class. Physical definitions remain shared until explicitly replaced. Profiles and installed systems are independent choices.

In [ ]:
project = archetype.with_changes(
    archetype_id="project:study",
    display_name="Project study",
    notes=archetype.notes + ("Explicit project selection for demonstration.",),
)
assert project.construction_set is archetype.construction_set
assert project.display_name != archetype.display_name
assert project.hvac_template is None

SE.07 begins in 2022 and has one configuration. Climate zones classify the physical constructions; they do not duplicate them. Complete exchange preserves the actual authored inputs.

In [ ]:
import json

from epsm_data import archetype_from_dict, archetype_to_dict

assert catalogue.era_for_year(2022) == "SE.07"
assert catalogue.renovation_states("mf_se07") == ()
assert len(catalogue.search(era="SE.07")) == 12
assert slot.complete == report["complete"]
assert tuple(slot.missing_values) == tuple(report["missing_values"])
restored = archetype_from_dict(json.loads(json.dumps(archetype_to_dict(project))))
assert restored == project
assert catalogue.to_dict()["dataset_id"] == catalogue.dataset_id
print(project)